# 08. Experiencia externa

**Fuente:** `data/raw/experenciaexterna.csv`  
**Salida:** `data/processed/experiencia_externa.csv`

Historial laboral del personal fuera de ESPOL (cargo, institucion, fechas, tipo de relacion laboral). Los codigos CATEXPERIENCIA y ROLACADEMICOEXPERIENCIA se decodifican usando data/raw/diccionarioexperienciaexterna.txt.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('experenciaexterna.csv', low_memory=False)
df.head()

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'experiencia_externa')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df)

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['FECHADESDE', 'FECHAHASTA'])
df = pc.castear_enteros(df, ['IDHISTORIALABORAL', 'IDPERSONA', 'IDPAIS'])

## 6. Decodificación de catálogos

Códigos definidos en `data/raw/diccionarioexperienciaexterna.txt`: CATEXPERIENCIA (PC=Por clasificar, AD=Administrativa, AC=Académica) y ROLACADEMICO (PR=Profesor, FA=Facilitador, PA=Personal de apoyo, AY=Ayudante). ROLACADEMICOEXPERIENCIA ya viene con el texto completo en el CSV de origen.

In [ ]:
df = pc.decodificar_experiencia_externa(df)
df[['CATEXPERIENCIA', 'CATEXPERIENCIA_DESC', 'ROLACADEMICO', 'ROLACADEMICO_DESC', 'ROLACADEMICOEXPERIENCIA']].drop_duplicates().head(10)

## Gráficos exploratorios

Vistas rápidas para apoyar la construcción del catálogo de variables del perfil (Fase 1-2 de la metodología): estacionalidad/tendencia temporal, categorías dominantes y forma de la distribución de las variables numéricas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Categoría de la experiencia externa** (académica, administrativa, por clasificar).

In [ ]:
pc.grafico_barras(df['CATEXPERIENCIA_DESC'], 'Experiencia externa por categoría', horizontal=False)

**Rol académico** declarado en la experiencia externa (solo ~9% de los registros lo especifica).

In [ ]:
pc.grafico_barras(df['ROLACADEMICO_DESC'], 'Experiencia externa por rol académico', horizontal=False)

**Duración de la experiencia externa**, en años, por registro.

In [ ]:
duracion_anios = (df['FECHAHASTA'] - df['FECHADESDE']).dt.days / 365.25
pc.grafico_histograma(duracion_anios, 'Duración de la experiencia externa (años)', bins=30)

**Países donde más experiencia externa se registra** (fuera de Ecuador incluido).

In [ ]:
pc.grafico_barras(df['PAIS'], 'Top 10 países de experiencia externa', top=10)

## 7. Verificación final

In [ ]:
pc.resumen(df, 'experiencia_externa (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'experiencia_externa.csv')